# 03 — Análise em SQL

As mesmas perguntas dos gráficos, respondidas em **SQL** sobre um banco SQLite.

- `src/banco.py` monta `data/ufc.db` a partir dos CSVs limpos, seguindo `sql/schema.sql`: os CSVs (uma linha por lutador em cada luta) viram cinco tabelas — `eventos` (com a data de cada evento) → `lutas` → `desempenho` → `desempenho_round`, mais `lutadores` (altura, envergadura, base, nascimento) ligada a `desempenho`.
- As consultas ficam em `sql/consultas.sql`, cada uma com um `-- nome:`; `src/consultas.py` as lê e executa.
- Cada resposta é **conferida com Pandas**: se os dois caminhos não baterem, a célula falha.

In [1]:
import sys
sys.path.append("..")

import sqlite3
import pandas as pd

from src.banco import criar_banco, ARQUIVO_BANCO
from src.consultas import carregar_consultas, consultar

# Recria o banco a partir dos CSVs (rápido; garante que está em dia com os dados)
for tabela, linhas in criar_banco().items():
    print(f"{tabela}: {linhas} linhas")

consultas = carregar_consultas()
df = pd.read_csv("../data/processed/dataset_final_resumo_limpo.csv")  # para conferência


def mostrar(nome):
    """Imprime o SQL da consulta e devolve o resultado."""
    print(consultas[nome], end="\n\n")
    return consultar(nome)

eventos: 795 linhas
lutas: 8974 linhas
lutadores: 2776 linhas
desempenho: 17948 linhas
desempenho_round: 42122 linhas


## O modelo

Tabelas e quantidade de linhas. A chave de `desempenho` é `(luta_id, lutador)`; cada luta tem exatamente dois registros.

In [2]:
with sqlite3.connect(ARQUIVO_BANCO) as conexao:
    tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid", conexao)["name"]
    print(pd.DataFrame({t: [pd.read_sql_query(f"SELECT COUNT(*) FROM {t}", conexao).iloc[0, 0]] for t in tabelas}, index=["linhas"]).T)
    print("Lutas com número de lutadores diferente de 2:",
          pd.read_sql_query("SELECT COUNT(*) FROM (SELECT luta_id FROM desempenho GROUP BY luta_id HAVING COUNT(*) <> 2)", conexao).iloc[0, 0])

                  linhas
eventos              795
lutas               8974
lutadores           2776
desempenho         17948
desempenho_round   42122
Lutas com número de lutadores diferente de 2: 0


## 1. Como as lutas terminam

In [3]:
metodos = mostrar("metodos_vitoria")
metodos

-- Como as lutas terminam? Uma linha por luta, então não há contagem dobrada.
SELECT
    metodo,
    COUNT(*)                                            AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)  AS pct
FROM lutas
GROUP BY metodo
ORDER BY lutas DESC;



,metodo,lutas,pct
0,Decision - Unanimous,3212,35.8
1,KO/TKO,2867,31.9
2,Submission,1736,19.3
3,Decision - Split,836,9.3
4,Decision - Majority,105,1.2
5,TKO - Doctor's Stoppage,100,1.1
6,Overturned,59,0.7
7,Could Not Continue,33,0.4
8,DQ,24,0.3
9,Other,2,0.0


In [4]:
# Conferência: mesma contagem do gráfico metodos_vitoria.png (uma linha por luta)
esperado = df.drop_duplicates("Fight_URL")["Method"].value_counts()
assert metodos.set_index("metodo")["lutas"].to_dict() == esperado.to_dict()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 2. O que separa vencedores de perdedores

In [5]:
vp = mostrar("vencedores_perdedores")
vp

-- O que separa quem vence de quem perde? Médias por luta de cada lado.
WITH medias AS (
    SELECT
        resultado,
        AVG(sig_acertados)            AS golpes_sig,
        AVG(quedas)                   AS quedas,
        AVG(controle_seg) / 60.0      AS controle_min,
        AVG(tent_finalizacao)         AS tent_finalizacao,
        AVG(kd)                       AS knockdowns
    FROM desempenho
    WHERE resultado IN ('W', 'L')
    GROUP BY resultado
)
SELECT
    ROUND(v.golpes_sig, 1)                         AS golpes_sig_v,
    ROUND(p.golpes_sig, 1)                         AS golpes_sig_p,
    ROUND(v.golpes_sig / p.golpes_sig, 1)          AS golpes_sig_razao,
    ROUND(v.quedas / p.quedas, 1)                  AS quedas_razao,
    ROUND(v.controle_min / p.controle_min, 1)      AS controle_razao,
    ROUND(v.tent_finalizacao / p.tent_finalizacao, 1) AS finalizacao_razao,
    ROUND(v.knockdowns / p.knockdowns, 1)          AS knockdowns_razao
FROM medias v
CROSS JOIN medias p


,golpes_sig_v,golpes_sig_p,golpes_sig_razao,quedas_razao,controle_razao,finalizacao_razao,knockdowns_razao
0,44.6,29.5,1.5,2.2,2.2,2.2,6.4


In [6]:
medias = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Sig_str_landed", "KD", "Td_landed"]].mean()
assert round(medias.loc["W", "KD"] / medias.loc["L", "KD"], 1) == vp.loc[0, "knockdowns_razao"]
assert round(medias.loc["W", "Td_landed"] / medias.loc["L", "Td_landed"], 1) == vp.loc[0, "quedas_razao"]
assert round(medias.loc["W", "Sig_str_landed"], 1) == vp.loc[0, "golpes_sig_v"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 3. Taxa de vitória (mín. 6 lutas)

`RANK()` deixa explícitos os empates: vários invictos dividem a 1ª posição.

In [7]:
taxa = mostrar("taxa_vitoria")
taxa

-- Top 10 por taxa de vitória, com amostra mínima de 6 lutas (NC não entra na conta).
-- RANK() mostra os empates: vários invictos dividem a 1ª posição.
SELECT
    RANK() OVER (ORDER BY 1.0 * SUM(resultado = 'W') / COUNT(*) DESC) AS posicao,
    lutador,
    SUM(resultado = 'W')                                  AS vitorias,
    SUM(resultado = 'L')                                  AS derrotas,
    SUM(resultado = 'D')                                  AS empates,
    ROUND(100.0 * SUM(resultado = 'W') / COUNT(*), 1)     AS taxa_pct
FROM desempenho
WHERE resultado IN ('W', 'L', 'D')
GROUP BY lutador
HAVING COUNT(*) >= 6
ORDER BY taxa_pct DESC, vitorias DESC
LIMIT 10;



,posicao,lutador,vitorias,derrotas,empates,taxa_pct
0,1,Khabib Nurmagomedov,13,0,0,100.0
1,1,Natalia Silva,9,0,0,100.0
2,1,Farid Basharat,7,0,0,100.0
3,1,Michael Morales,7,0,0,100.0
4,1,Muhammad Mokaev,7,0,0,100.0
5,1,Shavkat Rakhmonov,7,0,0,100.0
6,1,Navajo Stirling,6,0,0,100.0
7,1,Quillan Salkilld,6,0,0,100.0
8,1,Zabit Magomedsharipov,6,0,0,100.0
9,10,Jon Jones,22,1,0,95.7


In [8]:
cartel = pd.crosstab(df["Fighter"], df["Resultado"]).reindex(columns=["W", "L", "D"], fill_value=0)
cartel["lutas"] = cartel.sum(axis=1)
cartel = cartel[cartel["lutas"] >= 6]
cartel["taxa"] = cartel["W"] / cartel["lutas"] * 100
esperado = cartel.sort_values(["taxa", "W"], ascending=False).head(10)
assert list(taxa["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 4. Precisão de quedas (mín. 15 tentativas e 3 lutas)

In [9]:
quedas = mostrar("precisao_quedas")
quedas

-- Top 10 por precisão de quedas: pelo menos 15 tentativas E 3 lutas no período.
SELECT
    lutador,
    SUM(quedas)                                           AS conseguidas,
    SUM(quedas_tentadas)                                  AS tentadas,
    COUNT(*)                                              AS lutas,
    ROUND(100.0 * SUM(quedas) / SUM(quedas_tentadas), 1)  AS precisao_pct
FROM desempenho
GROUP BY lutador
HAVING SUM(quedas_tentadas) >= 15 AND COUNT(*) >= 3
ORDER BY precisao_pct DESC, conseguidas DESC
LIMIT 10;



,lutador,conseguidas,tentadas,lutas,precisao_pct
0,Ronnie Lawrence,15,18,4,83.3
1,Nordine Taleb,16,21,12,76.2
2,Taila Santos,15,20,7,75.0
3,Robbie Lawler,17,23,25,73.9
4,Georges St-Pierre,90,122,22,73.8
5,Yan Xiaonan,11,15,14,73.3
6,Bartosz Fabinski,24,34,6,70.6
7,Kevin Randleman,21,30,7,70.0
8,Jonathan Goulet,14,20,9,70.0
9,Travis Browne,11,16,17,68.8


In [10]:
g = df.groupby("Fighter").agg(c=("Td_landed", "sum"), t=("Td_attempted", "sum"), n=("Fight_URL", "nunique"))
g = g[(g["t"] >= 15) & (g["n"] >= 3)]
g["p"] = g["c"] / g["t"] * 100
esperado = g.sort_values(["p", "c"], ascending=False).head(10)
assert list(quedas["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 5. Em que round terminam nocautes e finalizações

Percentual **dentro de cada método** com `SUM(COUNT(*)) OVER (PARTITION BY tipo)`.

In [11]:
rounds = mostrar("round_final_por_metodo")
rounds

-- Em que round terminam nocautes e finalizações? Percentual dentro de cada método
-- (função de janela particionada por método). Rounds 4 e 5 agrupados.
WITH finalizadas AS (
    SELECT
        CASE
            WHEN metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%' THEN 'Nocaute'
            WHEN metodo LIKE '%Submission%' THEN 'Finalização'
        END AS tipo,
        CASE WHEN round_final >= 4 THEN '4º ou 5º' ELSE round_final || 'º' END AS round_grupo
    FROM lutas
)
SELECT
    tipo,
    round_grupo,
    COUNT(*)                                                               AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY tipo), 1)    AS pct_do_metodo
FROM finalizadas
WHERE tipo IS NOT NULL
GROUP BY tipo, round_grupo
ORDER BY tipo DESC, round_grupo;



,tipo,round_grupo,lutas,pct_do_metodo
0,Nocaute,1º,1607,54.2
1,Nocaute,2º,877,29.6
2,Nocaute,3º,423,14.3
3,Nocaute,4º ou 5º,60,2.0
4,Finalização,1º,885,51.0
5,Finalização,2º,548,31.6
6,Finalização,3º,276,15.9
7,Finalização,4º ou 5º,27,1.6


In [12]:
lutas_df = df.drop_duplicates("Fight_URL")
for tipo, padrao in [("Nocaute", "KO/TKO|TKO - Doctor"), ("Finalização", "Submission")]:
    do_metodo = lutas_df[lutas_df["Method"].str.contains(padrao, na=False)]
    esperado = round((do_metodo["Final_Round"] == 1).mean() * 100, 1)
    assert rounds.query("tipo == @tipo and round_grupo == '1º'")["pct_do_metodo"].item() == esperado
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 6. De onde saem os golpes de vencedores e perdedores

In [13]:
origem = mostrar("origem_golpes")
origem

-- De onde saem os golpes de vencedores e perdedores (distância, clinch, chão)?
SELECT
    CASE resultado WHEN 'W' THEN 'Vencedores' ELSE 'Perdedores' END  AS grupo,
    ROUND(100.0 * SUM(distancia) / SUM(sig_acertados), 1)            AS distancia_pct,
    ROUND(100.0 * SUM(clinch)    / SUM(sig_acertados), 1)            AS clinch_pct,
    ROUND(100.0 * SUM(chao)      / SUM(sig_acertados), 1)            AS chao_pct
FROM desempenho
WHERE resultado IN ('W', 'L')
GROUP BY resultado
ORDER BY grupo DESC;



,grupo,distancia_pct,clinch_pct,chao_pct
0,Vencedores,69.4,13.0,17.6
1,Perdedores,80.8,13.3,5.9


In [14]:
wl = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Ground_landed", "Sig_str_landed"]].sum()
chao = (wl["Ground_landed"] / wl["Sig_str_landed"] * 100).round(1)
assert origem.set_index("grupo").loc["Vencedores", "chao_pct"] == chao["W"]
assert origem.set_index("grupo").loc["Perdedores", "chao_pct"] == chao["L"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 7. Quem mais arrisca golpes (por minuto de luta)

`JOIN` com `lutas` para usar a duração real de cada luta.

In [15]:
volume = mostrar("volume_por_minuto")
volume

-- Quem mais arrisca golpes? Tentativas por minuto de luta (JOIN com lutas para a duração),
-- lutadores com pelo menos 4 lutas.
SELECT
    d.lutador,
    COUNT(*)                                                       AS lutas,
    ROUND(SUM(d.sig_tentados) / (SUM(l.duracao_seg) / 60.0), 1)    AS tentados_por_min,
    ROUND(100.0 * SUM(d.sig_acertados) / SUM(d.sig_tentados), 1)   AS precisao_pct
FROM desempenho AS d
JOIN lutas AS l USING (luta_id)
GROUP BY d.lutador
HAVING COUNT(*) >= 4
ORDER BY tentados_por_min DESC
LIMIT 10;



,lutador,lutas,tentados_por_min,precisao_pct
0,Martin Bravo,4,19.4,39.5
1,Leslie Smith,7,18.7,40.3
2,Carlos Leal,4,18.6,47.0
3,Carli Judice,5,18.1,48.1
4,Yazmin Jauregui,6,17.7,41.0
5,Djorden Santos,4,16.9,39.5
6,Esteban Ribovics,9,16.9,44.0
7,Josh Hokit,4,16.1,63.8
8,Marco Polo Reyes,8,16.1,37.3
9,Ryan Janes,4,15.9,48.2


## 8. Árbitros

O árbitro não escolhe o método: a taxa reflete as lutas que cada um recebe.

In [16]:
arbitros = mostrar("arbitros")
arbitros

-- Os 10 árbitros com mais lutas e a % delas encerrada antes da decisão.
SELECT
    arbitro,
    COUNT(*)                                                                    AS lutas,
    ROUND(100.0 * SUM(metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%'
                      OR metodo LIKE '%Submission%') / COUNT(*), 1)             AS pct_antes_da_decisao
FROM lutas
GROUP BY arbitro
ORDER BY lutas DESC
LIMIT 10;



,arbitro,lutas,pct_antes_da_decisao
0,Herb Dean,1280,55.5
1,John McCarthy,661,65.2
2,Marc Goddard,633,49.9
3,Dan Miragliotta,512,55.3
4,Jason Herzog,498,49.8
5,Chris Tognoni,473,48.6
6,Mark Smith,463,49.7
7,Keith Peterson,411,45.3
8,Mario Yamasaki,392,56.9
9,Yves Lavigne,265,50.6


## 9. Golpes por round

Tende a subir por viés de sobrevivência: só lutas longas chegam aos rounds finais.

In [17]:
mostrar("golpes_por_round")

-- Média de golpes significativos acertados por round (tende a subir por viés de sobrevivência:
-- só lutas longas chegam aos rounds finais).
SELECT
    round,
    COUNT(*)                     AS registros,
    ROUND(AVG(sig_acertados), 1) AS media_golpes_sig
FROM desempenho_round
GROUP BY round
ORDER BY round;

,round,registros,media_golpes_sig
0,1,17906,14.4
1,2,12844,16.3
2,3,9882,17.1
3,4,796,18.1
4,5,694,18.6


## 10. Canhoto × ortodoxo

*Self-join* de `desempenho`: a linha do vencedor encontra a do perdedor da mesma luta, e cada uma é ligada à base do lutador em `lutadores`. Os confrontos vêm de uma lista em `VALUES`, que fixa a ordem e qual lado é contado.

In [18]:
bases = mostrar("confronto_bases")
bases

-- Canhoto x ortodoxo: em lutas com vencedor entre bases diferentes, quantas o 1º lado venceu.
-- Self-join de desempenho (linha do vencedor com a do perdedor da mesma luta), cada uma ligada
-- à base do lutador em lutadores. Os confrontos são listados em VALUES para fixar a ordem e o lado.
WITH duelos AS (
    SELECT lv.base AS base_vencedor, lp.base AS base_perdedor
    FROM desempenho AS v
    JOIN desempenho AS p  ON p.luta_id = v.luta_id AND p.resultado = 'L'
    JOIN lutadores  AS lv ON lv.url = v.lutador_url
    JOIN lutadores  AS lp ON lp.url = p.lutador_url
    WHERE v.resultado = 'W' AND lv.base <> lp.base  -- base NULL fica de fora pela comparação
),
confrontos (ordem, lado_a, lado_b) AS (
    VALUES (1, 'Southpaw', 'Orthodox'), (2, 'Switch', 'Orthodox'), (3, 'Switch', 'Southpaw')
)
SELECT
    c.lado_a || ' x ' || c.lado_b                                         AS confronto,
    COUNT(*)                                                              AS lutas,
    SUM(d.base_v

,confronto,lutas,vitorias_lado_a,pct_lado_a
0,Southpaw x Orthodox,2470,1317,53.3
1,Switch x Orthodox,726,377,51.9
2,Switch x Southpaw,197,107,54.3


In [19]:
# Conferência: mesma contagem do gráfico base_canhoto_vs_ortodoxo.png, com o merge por Fighter_URL em Pandas
lutadores = pd.read_csv("../data/processed/lutadores.csv")
com_base = df.merge(lutadores[["Fighter_URL", "Stance"]], on="Fighter_URL")
duelos = pd.concat([
    com_base[com_base["Resultado"] == "W"].set_index("Fight_URL")["Stance"].rename("v"),
    com_base[com_base["Resultado"] == "L"].set_index("Fight_URL")["Stance"].rename("p"),
], axis=1, join="inner").dropna()
for linha in bases.itertuples():
    a, b = linha.confronto.split(" x ")
    no_confronto = ((duelos["v"] == a) & (duelos["p"] == b)) | ((duelos["v"] == b) & (duelos["p"] == a))
    assert (linha.lutas, linha.vitorias_lado_a) == (no_confronto.sum(), (no_confronto & (duelos["v"] == a)).sum())
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 11. Disputas de cinturão × demais lutas

A CTE soma os dois lutadores de cada luta; depois, um `GROUP BY` pela flag `disputa_titulo`. Ritmo por minuto e por lutador, porque as disputas são de 5 rounds.

In [20]:
titulo = mostrar("disputas_titulo")
titulo

-- Disputas de cinturão x demais lutas. A CTE soma os dois lutadores de cada luta; o ritmo é por
-- minuto de luta e por lutador (duração x 2), porque as disputas têm 5 rounds e duram mais.
WITH por_luta AS (
    SELECT l.luta_id, l.disputa_titulo, l.duracao_seg, l.metodo, l.bonus_luta, l.bonus_performance,
           SUM(d.sig_acertados) AS golpes,
           SUM(d.kd)            AS kd
    FROM lutas AS l
    JOIN desempenho AS d USING (luta_id)
    GROUP BY l.luta_id
)
SELECT
    CASE disputa_titulo WHEN 1 THEN 'Disputa de cinturão' ELSE 'Demais lutas' END  AS tipo,
    COUNT(*)                                                                      AS lutas,
    ROUND(AVG(duracao_seg) / 60.0, 1)                                             AS duracao_min,
    ROUND(SUM(golpes) / (SUM(duracao_seg) / 60.0 * 2), 2)                         AS golpes_por_min,
    ROUND(SUM(kd) * 15 / (SUM(duracao_seg) / 60.0 * 2), 2)                        AS kd_por_15min,
    ROUND(100.0 * AVG(metodo LIKE '

,tipo,lutas,duracao_min,golpes_por_min,kd_por_15min,nocaute_pct,finalizacao_pct,decisao_pct,luta_da_noite_pct,performance_pct
0,Disputa de cinturão,503,14.7,3.28,0.26,37.6,20.9,40.4,15.3,28.6
1,Demais lutas,8471,10.3,3.52,0.31,32.8,19.3,46.6,6.3,19.0


In [21]:
# Conferência: mesmos números do gráfico disputas_de_cinturao.png, calculados em Pandas
from src.limpeza import adicionar_duracao  # duração pelo formato de tempo, como no banco
por_luta = adicionar_duracao(df, "dur").groupby("Fight_URL").agg(
    titulo=("Title_Bout", "first"), dur=("dur", "first"), golpes=("Sig_str_landed", "sum"), bonus=("Fight_Bonus", "first"))
g = por_luta.groupby("titulo").agg(lutas=("dur", "size"), dur=("dur", "sum"), golpes=("golpes", "sum"), bonus=("bonus", "mean"))
sql = titulo.assign(titulo=(titulo["tipo"] == "Disputa de cinturão").astype(int)).set_index("titulo")
g = g.reindex(sql.index)  # mesma ordem do SQL (disputas primeiro)
assert (sql["lutas"] == g["lutas"]).all()
assert (sql["golpes_por_min"] == (g["golpes"] / (g["dur"] / 60 * 2)).round(2)).all()
assert (sql["luta_da_noite_pct"] == (g["bonus"] * 100).round(1)).all()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 12. Finalizações por categoria de peso

`HAVING` aplica a amostra mínima (20 lutas) depois do agrupamento, `RANK()` numera as categorias e uma subconsulta escalar traz a média geral para cada linha.

In [22]:
categorias = mostrar("finalizacoes_por_categoria")
categorias

-- Lutas encerradas antes da decisão, por categoria de peso: nocaute (inclui TKO médico) e
-- finalização. Fora o peso casado (acordo entre os lutadores) e o peso aberto (torneios dos
-- primeiros UFCs), que não são divisões, e, via HAVING, categorias com menos de 20 lutas.
-- media_geral: todas as lutas fora esses dois.
WITH por_categoria AS (
    SELECT
        categoria,
        COUNT(*)                                                               AS lutas,
        100.0 * AVG(metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%')     AS nocaute_pct,
        100.0 * AVG(metodo LIKE '%Submission%')                                AS finalizacao_pct
    FROM lutas
    WHERE categoria NOT IN ('Catch Weight', 'Open Weight')
    GROUP BY categoria
    HAVING COUNT(*) >= 20
)
SELECT
    RANK() OVER (ORDER BY nocaute_pct + finalizacao_pct DESC)  AS posicao,
    categoria,
    lutas,
    ROUND(nocaute_pct, 1)                                      AS nocaute_pct,
    ROUND(finalizacao_pct, 1)

,posicao,categoria,lutas,nocaute_pct,finalizacao_pct,antes_da_decisao_pct,media_geral
0,1,Heavyweight,796,51.6,14.4,66.1,51.9
1,2,Light Heavyweight,772,45.3,16.6,61.9,51.9
2,3,Middleweight,1187,37.9,21.3,59.2,51.9
3,4,Lightweight,1497,30.2,21.7,51.9,51.9
4,5,Welterweight,1422,33.4,18.4,51.8,51.9
5,6,Featherweight,897,29.9,16.5,46.4,51.9
6,7,Bantamweight,812,26.1,19.7,45.8,51.9
7,8,Flyweight,434,24.0,21.4,45.4,51.9
8,9,Women's Featherweight,30,23.3,20.0,43.3,51.9
9,10,Women's Bantamweight,255,22.0,17.3,39.2,51.9


In [23]:
# Conferência: mesma conta do gráfico finalizacoes_por_categoria.png (uma linha por luta, sem peso casado nem aberto)
lutas_cat = df.drop_duplicates("Fight_URL")
lutas_cat = lutas_cat[~lutas_cat["Weight_Class"].isin(["Catch Weight", "Open Weight"])].assign(
    nocaute=lambda x: x["Method"].str.contains("KO/TKO|TKO - Doctor", na=False),
    finalizacao=lambda x: x["Method"].str.contains("Submission", na=False))
g = lutas_cat.groupby("Weight_Class").agg(lutas=("Fight_URL", "size"), nocaute=("nocaute", "mean"), finalizacao=("finalizacao", "mean"))
g = g[g["lutas"] >= 20]
sql = categorias.set_index("categoria")
assert set(sql.index) == set(g.index)
g = g.reindex(sql.index)  # mesma ordem do SQL (por posição)
assert (sql["lutas"] == g["lutas"]).all()
assert (sql["antes_da_decisao_pct"] == ((g["nocaute"] + g["finalizacao"]) * 100).round(1)).all()
assert sql["media_geral"].iloc[0] == round((lutas_cat["nocaute"] | lutas_cat["finalizacao"]).mean() * 100, 1)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 13. Idade × envergadura

Idade na data do evento com `julianday()` (diferença em dias ÷ 365,25). *Self-join* do vencedor com o perdedor de cada luta, `CASE` para a faixa de diferença e `AVG` da condição (0/1) para a taxa. A linha `todas` vem de um `UNION ALL` que repete as lutas com a faixa 0.

In [24]:
fisico = mostrar("idade_envergadura")
fisico

-- Em lutas com vencedor, quem tinha a vantagem venceu com que frequência? Idade na data do
-- evento com julianday(); envergadura em polegadas (1" = 2,54 cm). Self-join do vencedor com o
-- perdedor da mesma luta; cada diferença cai numa faixa (CASE) e AVG da condição dá a taxa.
-- ordem 0 = todas as lutas do painel. Diferença zero fica de fora.
WITH fisico AS (
    SELECT
        d.luta_id,
        d.resultado,
        (julianday(e.data) - julianday(f.nascimento)) / 365.25  AS idade,
        ROUND(f.envergadura_cm / 2.54, 1)                       AS envergadura_pol
    FROM desempenho AS d
    JOIN lutas      AS l USING (luta_id)
    JOIN eventos    AS e USING (evento_id)
    JOIN lutadores  AS f ON f.url = d.lutador_url
    WHERE d.resultado IN ('W', 'L')
),
duelos AS (
    SELECT
        v.idade - p.idade                                       AS dif_idade,  -- negativo = vencedor mais novo
        ROUND(v.envergadura_pol - p.envergadura_pol, 1)         AS dif_env     -- positivo = 

,vantagem,ordem,diferenca,lutas,pct_vantagem_venceu
0,Mais novo,0,todas,8689,57.1
1,Mais novo,1,até 2 anos,2565,52.2
2,Mais novo,2,2 a 5 anos,3124,55.9
3,Mais novo,3,5 a 8 anos,1841,59.5
4,Mais novo,4,mais de 8 anos,1159,67.3
5,Mais envergadura,0,todas,6756,52.0
6,Mais envergadura,1,"1""",1837,50.1
7,Mais envergadura,2,"2""",1602,50.0
8,Mais envergadura,3,"3""",1219,52.1
9,Mais envergadura,4,"4"" ou mais",2098,55.3


In [25]:
# Conferência: mesmas faixas do gráfico idade_vs_envergadura.png, com pd.cut sobre as datas em Pandas
datas_ev = pd.read_csv("../data/processed/eventos.csv")[["Event_URL", "Event_Date"]]
f = df.merge(lutadores, on="Fighter_URL").merge(datas_ev, on="Event_URL")
f["idade"] = (pd.to_datetime(f["Event_Date"]) - pd.to_datetime(f["DOB"])).dt.days / 365.25
f["env"] = (f["Reach_cm"] / 2.54).round(1)
v = f[f["Resultado"] == "W"].set_index("Fight_URL")[["idade", "env"]]
p = f[f["Resultado"] == "L"].set_index("Fight_URL")[["idade", "env"]]
d = v.join(p, lsuffix="_v", rsuffix="_p", how="inner")
dif_idade, dif_env = d["idade_v"] - d["idade_p"], (d["env_v"] - d["env_p"]).round(1)
# Como o SQL (WHERE dif <> 0): sem dado físico (NULL) ou diferença zero ficam de fora
dif_idade = dif_idade[dif_idade.notna() & (dif_idade != 0)]
dif_env = dif_env[dif_env.notna() & (dif_env != 0)]
paineis = {
    "Mais novo": ((dif_idade < 0).groupby(pd.cut(dif_idade.abs(), [0, 2, 5, 8, 99], labels=[1, 2, 3, 4]), observed=True), dif_idade < 0),
    "Mais envergadura": ((dif_env > 0).groupby(pd.cut(dif_env.abs(), [0, 1, 2, 3, 99], labels=[1, 2, 3, 4]), observed=True), dif_env > 0),
}
for vantagem, (por_faixa, todas) in paineis.items():
    esperado = por_faixa.agg(["size", "mean"])
    esperado.index = esperado.index.astype(int)  # faixas do pd.cut (categóricas) -> 1..4, como a coluna ordem
    esperado.loc[0] = [len(todas), todas.mean()]
    esperado = esperado.sort_index()
    sql = fisico[fisico["vantagem"] == vantagem].set_index("ordem").sort_index()
    assert list(sql.index) == list(esperado.index)
    assert list(sql["lutas"]) == list(esperado["size"].astype(int))
    assert list(sql["pct_vantagem_venceu"]) == list((esperado["mean"] * 100).round(1))
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 14. O que faz uma Luta da Noite

Duas CTEs: `por_luta` soma os dois lutadores e `perdedor` traz a linha de quem perdeu. O `LEFT JOIN` deixa `NULL` nas lutas sem vencedor, e como o `AVG` ignora `NULL`, "ida e volta" e "equilibradas" contam só lutas com vencedor sem precisar de filtro.

In [26]:
ln = mostrar("luta_da_noite")
ln

-- O que as Lutas da Noite têm de diferente. por_luta soma os dois lutadores; perdedor traz a linha
-- de quem perdeu (LEFT JOIN: lutas sem vencedor ficam com NULL e o AVG as ignora, então "ida e
-- volta" e "equilibradas" contam só lutas com vencedor). Ritmo e controle por minuto de luta.
WITH por_luta AS (
    SELECT l.luta_id, l.bonus_luta, l.duracao_seg, l.metodo, l.round_final,
           SUM(d.sig_acertados) AS golpes,
           SUM(d.kd)            AS kd,
           SUM(d.controle_seg)  AS controle
    FROM lutas AS l
    JOIN desempenho AS d USING (luta_id)
    GROUP BY l.luta_id
),
perdedor AS (
    SELECT luta_id, sig_acertados AS golpes_perdedor, kd AS kd_perdedor
    FROM desempenho
    WHERE resultado = 'L'
)
SELECT
    CASE pl.bonus_luta WHEN 1 THEN 'Luta da Noite' ELSE 'Demais lutas' END  AS tipo,
    COUNT(*)                                                              AS lutas,
    ROUND(SUM(pl.golpes) / (SUM(pl.duracao_seg) / 60.0 * 2), 2)           AS golpes_por_min

,tipo,lutas,golpes_por_min,com_knockdown_pct,ida_e_volta_pct,equilibradas_pct,controle_pct,decisao_pct,primeiro_round_pct
0,Luta da Noite,612,4.84,47.2,17.5,59.4,31.5,58.3,4.1
1,Demais lutas,8362,3.36,34.9,4.5,45.5,41.8,45.4,30.2


In [27]:
# Conferência: mesmas seis métricas do gráfico luta_da_noite.png, calculadas em Pandas
from src.limpeza import adicionar_duracao  # duração pelo formato de tempo, como no banco
por_luta = adicionar_duracao(df, "dur").groupby("Fight_URL").agg(
    premiada=("Fight_Bonus", "first"), dur=("dur", "first"), metodo=("Method", "first"),
    golpes=("Sig_str_landed", "sum"), kd=("KD", lambda s: s.sum(min_count=1)), ctrl=("Ctrl_seconds", "sum"))
perd = df[df["Resultado"] == "L"].set_index("Fight_URL")[["Sig_str_landed", "KD"]]
por_luta = por_luta.join(perd)
com_vencedor = por_luta.dropna(subset=["KD"])
g = pd.DataFrame({
    "golpes_por_min": por_luta.groupby("premiada").apply(lambda x: x["golpes"].sum() / (x["dur"].sum() / 60 * 2), include_groups=False).round(2),
    "com_knockdown_pct": (por_luta.groupby("premiada")["kd"].apply(lambda k: (k.dropna() > 0).mean()) * 100).round(1),  # AVG do SQL ignora NULL
    "ida_e_volta_pct": (com_vencedor.groupby("premiada")["KD"].apply(lambda k: (k > 0).mean()) * 100).round(1),
    "equilibradas_pct": (com_vencedor.groupby("premiada").apply(lambda x: (x["Sig_str_landed"] >= 0.4 * x["golpes"]).mean(), include_groups=False) * 100).round(1),
    "controle_pct": por_luta.groupby("premiada").apply(lambda x: x["ctrl"].sum() / x["dur"].sum() * 100, include_groups=False).round(1),
    "decisao_pct": (por_luta.groupby("premiada")["metodo"].apply(lambda m: m.str.contains("Decision", na=False).mean()) * 100).round(1),
})
sql = ln.assign(premiada=(ln["tipo"] == "Luta da Noite").astype(int)).set_index("premiada")
g = g.reindex(sql.index)  # mesma ordem do SQL (premiadas primeiro)
for coluna in g.columns:
    assert list(sql[coluna]) == list(g[coluna]), coluna
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 15. Consultas que só aparecem aqui

Perguntas que seriam trabalhosas em Pandas e ficam naturais com funções de janela.

### Destaque de cada evento

`ROW_NUMBER() OVER (PARTITION BY evento)` escolhe quem mais acertou golpes significativos numa luta, em cada evento.

In [28]:
mostrar("destaque_por_evento")

-- Em cada evento, quem acertou mais golpes significativos numa única luta?
-- ROW_NUMBER() por evento escolhe um só por evento; mostra os 10 maiores desses recordes.
WITH ranqueado AS (
    SELECT
        e.nome        AS evento,
        d.lutador,
        d.sig_acertados,
        ROW_NUMBER() OVER (PARTITION BY l.evento_id ORDER BY d.sig_acertados DESC, d.lutador) AS n
    FROM desempenho AS d
    JOIN lutas   AS l USING (luta_id)
    JOIN eventos AS e USING (evento_id)
)
SELECT evento, lutador, sig_acertados
FROM ranqueado
WHERE n = 1
ORDER BY sig_acertados DESC
LIMIT 10;



,evento,lutador,sig_acertados
0,UFC Fight Night: Holloway vs. Kattar,Max Holloway,445
1,UFC 231: Holloway vs. Ortega,Max Holloway,290
2,UFC Fight Night: Font vs. Vera,Rob Font,271
3,UFC Fight Night: Du Plessis vs. Usman,Tommy McMillen,252
4,UFC Fight Night: Vettori vs. Cannonier,Jared Cannonier,241
5,UFC 141: Lesnar vs Overeem,Nate Diaz,238
6,UFC 283: Teixeira vs. Hill,Jamahal Hill,232
7,UFC 299: O'Malley vs. Vera 2,Sean O'Malley,230
8,UFC Fight Night: Holloway vs. Rodriguez,Max Holloway,230
9,UFC 264: Poirier vs. McGregor 3,Sean O'Malley,230


### Maior sequência de vitórias

Problema clássico de *gaps and islands*: com as lutas de cada lutador em ordem de data, dentro de uma sequência de resultados iguais a diferença entre o número da luta na carreira e o número da luta entre as de mesmo resultado é constante — e vira o identificador da sequência.

In [29]:
mostrar("sequencia_vitorias")

-- Maior sequência de vitórias seguidas de cada lutador no período ("gaps and islands"):
-- a diferença entre dois ROW_NUMBER() é constante dentro de uma sequência de mesmo resultado.
-- As lutas de cada lutador são ordenadas pela data do evento.
WITH ordenado AS (
    SELECT
        d.lutador,
        d.resultado,
        ROW_NUMBER() OVER (PARTITION BY d.lutador ORDER BY e.data)
          - ROW_NUMBER() OVER (PARTITION BY d.lutador, d.resultado ORDER BY e.data) AS grupo
    FROM desempenho AS d
    JOIN lutas   AS l USING (luta_id)
    JOIN eventos AS e USING (evento_id)
),
sequencias AS (
    SELECT lutador, COUNT(*) AS vitorias_seguidas
    FROM ordenado
    WHERE resultado = 'W'
    GROUP BY lutador, grupo
)
SELECT lutador, MAX(vitorias_seguidas) AS maior_sequencia
FROM sequencias
GROUP BY lutador
ORDER BY maior_sequencia DESC, lutador
LIMIT 10;



,lutador,maior_sequencia
0,Islam Makhachev,17
1,Anderson Silva,16
2,Kamaru Usman,15
3,Merab Dvalishvili,14
4,Demetrious Johnson,13
5,Georges St-Pierre,13
6,Jon Jones,13
7,Khabib Nurmagomedov,13
8,Max Holloway,13
9,Alexander Volkanovski,12


### Como as lutas terminam, ano a ano

Usa a data de cada evento (`strftime('%Y', data)`). 2023 e o ano corrente são parciais: o recorte começa em abril de 2023.

In [30]:
por_ano = mostrar("lutas_por_ano")
por_ano

-- Como as lutas terminam a cada ano do período (strftime extrai o ano da data do evento).
-- 2023 e o ano corrente são parciais: o recorte começa em abril de 2023.
SELECT
    strftime('%Y', e.data)                                                          AS ano,
    COUNT(DISTINCT e.evento_id)                                                     AS eventos,
    COUNT(*)                                                                        AS lutas,
    ROUND(100.0 * SUM(l.metodo LIKE '%KO/TKO%' OR l.metodo LIKE 'TKO - Doctor%') / COUNT(*), 1) AS nocaute_pct,
    ROUND(100.0 * SUM(l.metodo LIKE '%Submission%') / COUNT(*), 1)                  AS finalizacao_pct,
    ROUND(100.0 * SUM(l.metodo LIKE 'Decision%') / COUNT(*), 1)                     AS decisao_pct
FROM lutas AS l
JOIN eventos AS e USING (evento_id)
GROUP BY ano
ORDER BY ano;



,ano,eventos,lutas,nocaute_pct,finalizacao_pct,decisao_pct
0,1994,3,31,35.5,64.5,0.0
1,1995,4,40,37.5,50.0,7.5
2,1996,5,43,62.8,25.6,11.6
3,1997,5,41,31.7,46.3,19.5
4,1998,3,25,36.0,32.0,32.0
5,1999,6,44,59.1,15.9,25.0
6,2000,6,43,32.6,25.6,39.5
7,2001,5,40,42.5,22.5,30.0
8,2002,7,53,54.7,15.1,28.3
9,2003,5,41,43.9,19.5,34.1


In [31]:
# Conferência: mesma contagem de lutas por ano feita em Pandas, juntando as datas de eventos.csv
datas = pd.read_csv("../data/processed/eventos.csv").set_index("Event_URL")["Event_Date"]
ano = df.drop_duplicates("Fight_URL")["Event_URL"].map(datas).str[:4]
assert por_ano.set_index("ano")["lutas"].to_dict() == ano.value_counts().to_dict()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem
